In [16]:
# Import Libraries
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib import colors

plt.rcParams.update({'figure.figsize': (11,9)})

# Load neighborhood geometries
gdf = gpd.read_file('Data/Processed/neighborhoods_final.gpkg')
# standardize geoid column name if present
def _find_geoid_col(df):
    for c in df.columns:
        if 'geoid' in c.lower() or 'geoid10' in c.lower():
            return c
    return None
gdf_geoid = _find_geoid_col(gdf)
if gdf_geoid is None:
    # try common alternatives
    if 'GEOID' in gdf.columns:
        gdf_geoid = 'GEOID'
if gdf_geoid is not None:
    gdf = gdf.rename(columns={gdf_geoid: 'geoid'})
else:
    gdf['geoid'] = gdf.index.astype(str)


In [17]:
# Import WBGT Index
# This file was produced in earlier notebooks; it contains neighborhood-level WBGT statistics by month.
try:
    wbgt = pd.read_csv('Outputs/Statistics/neighborhood_wbgt_by_month.csv')
except Exception:
    # fallback to processed data if present
    try:
        wbgt = pd.read_csv('Data/Processed/neighborhood_wbgt_by_month.csv')
    except Exception:
        wbgt = pd.DataFrame()

# find wbgt column and geoid column robustly
def find_col(df, keywords):
    for col in df.columns:
        name = col.lower()
        if any(k in name for k in keywords):
            return col
    return None

wbgt_geoid = find_col(wbgt, ['geoid','geo','tract','nta'])
wbgt_col = find_col(wbgt, ['wbgt','wbgt_mean','wbgt_avg','mean','value'])
if wbgt_geoid is None and 'geoid' in gdf.columns:
    # expect geography join later using gdf geoid
    wbgt_geoid = None

# If WBGT is monthly by neighborhood, compute long-term mean if needed
if not wbgt.empty:
    if wbgt_col is None:
        # try numeric columns as candidate
        numcols = wbgt.select_dtypes('number').columns.tolist()
        wbgt_col = numcols[0] if numcols else None
    # If there are month columns, compute row mean
    if wbgt_col is None and any(c.startswith('20') or c.startswith('m') for c in wbgt.columns):
        # compute mean across numeric cols as fallback
        wbgt['wbgt_mean'] = wbgt.select_dtypes('number').mean(axis=1)
        wbgt_col = 'wbgt_mean'
    # rename geoid if found
    if wbgt_geoid is not None:
        wbgt = wbgt.rename(columns={wbgt_geoid: 'geoid'})

# keep only geoid and wbgt_col
if not wbgt.empty and wbgt_col is not None:
    wbgt = wbgt[['geoid', wbgt_col]] if 'geoid' in wbgt.columns else wbgt[[wbgt_col]]
    wbgt = wbgt.rename(columns={wbgt_col: 'wbgt'})

wbgt.head()


""


In [18]:
# Import CDC's Social Vulnerability Index (SVI)
svi = pd.read_csv('Data/Processed/socialvulnerability.csv')
# find a sensible SVI index column (overall vulnerability)
svi_col = find_col(svi, ['svi','vulnerability','rpl_themes','rpl_them'])
if svi_col is None:
    # fallback to first numeric column that isn't an id
    num = [c for c in svi.select_dtypes('number').columns if 'id' not in c.lower()]
    svi_col = num[0] if num else None
# standardize geoid col name in SVI
svi_geoid = find_col(svi, ['geoid','geo','tract','nta'])
if svi_geoid is not None:
    svi = svi.rename(columns={svi_geoid: 'geoid'})
if svi_col is not None:
    svi = svi[['geoid', svi_col]] if 'geoid' in svi.columns else svi[[svi_col]]
    svi = svi.rename(columns={svi_col: 'svi'})
svi.head()


,svi
0,0.9384
1,0.6864
2,0.8324
3,0.8466
4,0.4773


In [19]:
# Import Infrastructure / Adaptive Capacity Index
# Earlier notebooks may have produced an adaptive capacity or infrastructure table; try plausible files.
candidates = [
    'Data/Processed/neighborhoods_attributes.csv',
    'Data/Processed/urban_design_final.csv',
    'Data/Processed/urban_design_final.gpkg'
]
adaptive = pd.DataFrame()
for f in candidates:
    try:
        if f.endswith('.gpkg'):
            adaptive = gpd.read_file(f)
        else:
            adaptive = pd.read_csv(f)
        break
    except Exception:
        adaptive = pd.DataFrame()

adaptive_col = None
if not adaptive.empty:
    adaptive_col = find_col(adaptive, ['adaptive','capacity','adapt','infra','infrastruct','design','index'])
    geoid_col = find_col(adaptive, ['geoid','geo','tract','nta'])
    if geoid_col is not None:
        adaptive = adaptive.rename(columns={geoid_col: 'geoid'})
    if adaptive_col is None:
        num = [c for c in adaptive.select_dtypes('number').columns if 'id' not in c.lower()]
        adaptive_col = num[0] if num else None
    if adaptive_col is not None:
        adaptive = adaptive[['geoid', adaptive_col]] if 'geoid' in adaptive.columns else adaptive[[adaptive_col]]
        adaptive = adaptive.rename(columns={adaptive_col: 'adaptive'})
adaptive.head()


,geoid,adaptive
0,BK0101,3
1,BK0102,3
2,BK0103,3
3,BK0104,3
4,BK0201,3


In [20]:
# Construct Composite Heat Vulnerability Index (CHVI)
# Merge data into the neighborhoods GeoDataFrame
df = gdf.copy()
# ensure geoid present in gdf
if 'geoid' not in df.columns:
    df['geoid'] = df.index.astype(str)

# merge SVI
if not svi.empty and 'geoid' in svi.columns:
    df = df.merge(svi, on='geoid', how='left')
else:
    df['svi'] = np.nan
# merge WBGT
if not wbgt.empty and 'geoid' in wbgt.columns:
    df = df.merge(wbgt, on='geoid', how='left')
else:
    # If wbgt has no geoid, attempt to align by index if lengths match
    if not wbgt.empty and len(wbgt)==len(df):
        df['wbgt'] = wbgt['wbgt'].values
    else:
        df['wbgt'] = np.nan
# merge adaptive/infrastructure
if not adaptive.empty and 'geoid' in adaptive.columns:
    df = df.merge(adaptive, on='geoid', how='left')
else:
    df['adaptive'] = np.nan

# Add NYC identifier: try geoid prefix (state FIPS 36) or borough column
def detect_in_nyc(row):
    try:
        geoid = str(row.get('geoid',''))
        if geoid.startswith('36'):
            return True
    except Exception:
        pass
    # try borough column
    for c in ['borough','Borough','BOROUGH','city','City']:
        if c in row.index:
            val = str(row[c]).lower()
            if any(b in val for b in ['manhattan','bronx','brooklyn','kings','queens','richmond','new york']):
                return True
    return False

df['in_nyc'] = df.apply(detect_in_nyc, axis=1)

# Handle NaNs: fill each component with its median (if available) to avoid NaNs affecting CHVI
for col in ['wbgt','svi','adaptive']:
    if col in df.columns:
        med = float(df[col].median(skipna=True)) if not df[col].dropna().empty else np.nan
        if not np.isnan(med):
            df[f'{col}_filled'] = df[col].fillna(med)
        else:
            df[f'{col}_filled'] = df[col]
    else:
        df[f'{col}_filled'] = np.nan

# Normalization helper (min-max). If column is all NaN, leave as NaN.
def minmax(s):
    s = pd.to_numeric(s, errors='coerce')
    if s.dropna().empty:
        return s*0 + np.nan
    mn = s.min()
    mx = s.max()
    if mx==mn:
        return (s - mn) * 0.0
    return (s - mn) / (mx - mn)

# Scale components using filled columns
df['wbgt_s'] = minmax(df['wbgt_filled'])
df['svi_s'] = minmax(df['svi_filled'])
# adaptive capacity: higher = more capacity -> less vulnerability, so invert after scaling
df['adaptive_s'] = minmax(df['adaptive_filled'])
df['adaptive_vuln_s'] = 1 - df['adaptive_s']

# Combine: simple average of the three scaled components (wbgt, svi, adaptive_vuln)
df['chvi'] = df[['wbgt_s','svi_s','adaptive_vuln_s']].mean(axis=1, skipna=True)

# Save full output for later notebooks/analysis
out_path = 'Data/Processed/neighborhoods_chvi.gpkg'
try:
    df.to_file(out_path, layer='neighborhoods_chvi', driver='GPKG')
except Exception:
    pass
try:
    df[['geoid','in_nyc','wbgt','svi','adaptive','wbgt_s','svi_s','adaptive_vuln_s','chvi']].to_csv('Data/Processed/neighborhoods_chvi.csv', index=False)
except Exception:
    pass

# Create a 20-row sample table (top by CHVI)
df_sample = df[['geoid','in_nyc','wbgt','svi','adaptive','wbgt_s','svi_s','adaptive_vuln_s','chvi']].copy()
df_sample = df_sample.sort_values('chvi', ascending=False).head(20)
try:
    df_sample.to_csv('Data/Processed/neighborhoods_chvi_sample.csv', index=False)
except Exception:
    pass

# show first rows of the full dataframe
df.head()


,geoid,BoroCode,BoroName,CountyFIPS,NTA2020,NTAName,NTAAbbrev,NTAType,CDTA2020,CDTAName,...,adaptive,in_nyc,wbgt_filled,svi_filled,adaptive_filled,wbgt_s,svi_s,adaptive_s,adaptive_vuln_s,chvi
0,BK0101,3,Brooklyn,047,BK0101,Greenpoint,Grnpt,0,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),...,3,False,NaN,NaN,3,NaN,NaN,0.5,0.5,0.5
1,BK0102,3,Brooklyn,047,BK0102,Williamsburg,Wllmsbrg,0,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),...,3,False,NaN,NaN,3,NaN,NaN,0.5,0.5,0.5
2,BK0103,3,Brooklyn,047,BK0103,South Williamsburg,SWllmsbrg,0,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),...,3,False,NaN,NaN,3,NaN,NaN,0.5,0.5,0.5
3,BK0104,3,Brooklyn,047,BK0104,East Williamsburg,EWllmsbrg,0,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),...,3,False,NaN,NaN,3,NaN,NaN,0.5,0.5,0.5
4,BK0201,3,Brooklyn,047,BK0201,Brooklyn Heights,BkHts,0,BK02,BK02 Downtown Brooklyn-Fort Greene (CD 2 Appro...,...,3,False,NaN,NaN,3,NaN,NaN,0.5,0.5,0.5


In [21]:
# Map visualization moved to notebook 06 — produce the CHVI sample table here
# Display the 20-row CHVI sample saved earlier
try:
    display(df_sample)
except Exception:
    print(df_sample.to_string(index=False))
print('\nSaved sample to Data/Processed/neighborhoods_chvi_sample.csv')


,geoid,in_nyc,wbgt,svi,adaptive,wbgt_s,svi_s,adaptive_vuln_s,chvi
148,MN1102,False,NaN,NaN,1,NaN,NaN,1.0,1.0
145,MN1001,False,NaN,NaN,1,NaN,NaN,1.0,1.0
144,MN0903,False,NaN,NaN,1,NaN,NaN,1.0,1.0
122,MN0202,False,NaN,NaN,1,NaN,NaN,1.0,1.0
123,MN0203,False,NaN,NaN,1,NaN,NaN,1.0,1.0
119,MN0102,False,NaN,NaN,1,NaN,NaN,1.0,1.0
118,MN0101,False,NaN,NaN,1,NaN,NaN,1.0,1.0
137,MN0702,False,NaN,NaN,1,NaN,NaN,1.0,1.0
136,MN0701,False,NaN,NaN,1,NaN,NaN,1.0,1.0
143,MN0902,False,NaN,NaN,1,NaN,NaN,1.0,1.0



Saved sample to Data/Processed/neighborhoods_chvi_sample.csv


In [22]:
# Summary statistics and quick checks
print('CHVI saved to Data/Processed/neighborhoods_chvi.gpkg and neighborhoods_chvi.csv')
print('Sample saved to Data/Processed/neighborhoods_chvi_sample.csv')
print('\nSample (20 rows):')
try:
    display(df_sample)
except Exception:
    print(df_sample.to_string(index=False))


CHVI saved to Data/Processed/neighborhoods_chvi.gpkg and neighborhoods_chvi.csv
Sample saved to Data/Processed/neighborhoods_chvi_sample.csv

Sample (20 rows):


,geoid,in_nyc,wbgt,svi,adaptive,wbgt_s,svi_s,adaptive_vuln_s,chvi
148,MN1102,False,NaN,NaN,1,NaN,NaN,1.0,1.0
145,MN1001,False,NaN,NaN,1,NaN,NaN,1.0,1.0
144,MN0903,False,NaN,NaN,1,NaN,NaN,1.0,1.0
122,MN0202,False,NaN,NaN,1,NaN,NaN,1.0,1.0
123,MN0203,False,NaN,NaN,1,NaN,NaN,1.0,1.0
119,MN0102,False,NaN,NaN,1,NaN,NaN,1.0,1.0
118,MN0101,False,NaN,NaN,1,NaN,NaN,1.0,1.0
137,MN0702,False,NaN,NaN,1,NaN,NaN,1.0,1.0
136,MN0701,False,NaN,NaN,1,NaN,NaN,1.0,1.0
143,MN0902,False,NaN,NaN,1,NaN,NaN,1.0,1.0
